In [1]:
import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
dsList = []

for year in range(2014, 2025):
    print(year)
    folder = f'/srv/cdx/hseo/Data/GLORYS12v1/PCF/{year:04d}/'
    #folder = f'/proj/cmip6/data/ocean_reanalysis/glorys12v1/{year:04d}/'
    ds = xr.open_mfdataset(glob(folder + '*.nc'))
    
    thetao = ds.thetao.sel(depth=slice(0,300))
    thetao = thetao.sel(latitude = 0, method='nearest')

    so = ds.so.sel(depth=slice(0,300))
    so = so.sel(latitude = 0, method='nearest')
    
    uo = ds.uo.sel(depth=slice(0,300))
    uo = uo.sel(latitude = 0, method='nearest')
    
    zos = ds.zos.sel(latitude = 0, method='nearest')
    
    mlotst = ds.mlotst.sel(latitude =0, method='nearest')
    
    ds.close()
    
    subds = xr.Dataset()
    subds['uo'] = uo
    subds['thetao'] = thetao
    subds['so'] = so
    subds['zos'] = zos
    subds['mld'] = mlotst
    
    dsList.append(subds)
    

2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [3]:
fullDS = xr.concat(dsList, dim='time').compute()

KeyboardInterrupt: 

In [ ]:
fullDS

In [ ]:
writeFname = '../../WPWP_GLORYS_data/equator_thetao_u_ssh_mld_2014_2024.nc'
new_lon = (fullDS['longitude'].values + 360) % 360
fullDS = fullDS.assign_coords(longitude=new_lon)
fullDS = fullDS.sortby('longitude')
fullDS = fullDS.sel(longitude=slice(106, 296))

# Write immediately
fullDS.to_netcdf(writeFname, unlimited_dims='time')